In [9]:
!pip install -q datasets pandas matplotlib
print("OK")

OK


In [10]:
from datasets import load_dataset

corpus  = load_dataset("maastrichtlawtech/bsard", name="corpus",    split="corpus")
q_train = load_dataset("maastrichtlawtech/bsard", name="questions", split="train")
q_test  = load_dataset("maastrichtlawtech/bsard", name="questions", split="test")

print(f"Corpus  : {len(corpus)} articles")
print(f"Q train : {len(q_train)} questions")
print(f"Q test  : {len(q_test)} questions")
print(f"Colonnes corpus    : {corpus.column_names}")
print(f"Colonnes questions : {q_train.column_names}")

README.md:   0%|          | 0.00/11.3k [00:00<?, ?B/s]

articles.csv:   0%|          | 0.00/30.1M [00:00<?, ?B/s]

Generating corpus split: 0 examples [00:00, ? examples/s]

questions_train.csv:   0%|          | 0.00/188k [00:00<?, ?B/s]

questions_synthetic.csv:   0%|          | 0.00/8.25M [00:00<?, ?B/s]

questions_test.csv:   0%|          | 0.00/46.1k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating synthetic split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Corpus  : 22633 articles
Q train : 886 questions
Q test  : 222 questions
Colonnes corpus    : ['id', 'reference', 'article', 'law_type', 'code', 'book', 'part', 'act', 'chapter', 'section', 'subsection', 'description']
Colonnes questions : ['id', 'category', 'subcategory', 'question', 'extra_description', 'article_ids']


In [11]:
import pandas as pd

df_corpus = corpus.to_pandas()

print("=== Exemple article ===")
print(df_corpus.iloc[0][["id", "code", "law_type", "article"]].to_string())

print(f"\n=== Statistiques ===")
print(f"Longueur moyenne des articles : {df_corpus['article'].str.len().mean():.0f} caractères")
print(f"Longueur max                  : {df_corpus['article'].str.len().max()} caractères")
print(f"\nTop 10 codes :")
print(df_corpus['code'].value_counts().head(10))
print(f"\nTypes de loi :")
print(df_corpus['law_type'].value_counts())

=== Exemple article ===
id                                                          1
code        Code Bruxellois de l'Air, du Climat et de la M...
law_type                                             regional
article     Le présent Code règle une matière visée à l'ar...

=== Statistiques ===
Longueur moyenne des articles : 880 caractères
Longueur max                  : 39566 caractères

Top 10 codes :
code
Code Réglementaire Wallon de l'Action sociale et de la Santé    2618
Code Judiciaire                                                 2285
Code de Droit Economique                                        2032
Code Civil                                                      1961
Code du Bien-être au Travail                                    1287
Code des Sociétés et des Associations                           1194
Code de la Démocratie Locale et de la Décentralisation          1159
Code Wallon de l'Action sociale et de la Santé                  1032
Code de la Navigation                

In [12]:
df_train = q_train.to_pandas()
df_test  = q_test.to_pandas()

def parse_ids(raw):
    if isinstance(raw, str):
        return [a.strip() for a in raw.split(",")]
    return [str(a) for a in raw]

print("=== Exemple question ===")
print(df_train.iloc[0])
print(f"\nNb moyen d'articles pertinents : {df_train['article_ids'].apply(parse_ids).apply(len).mean():.2f}")
print(f"\nTop catégories :")
print(df_train['category'].value_counts().head(10))

=== Exemple question ===
id                                                                1102
category                                                       Travail
subcategory                                     Travail et parentalité
question             Je suis travailleur salarié(e). Puis-je refuse...
extra_description                                 Pendant la grossesse
article_ids          22225,22226,22227,22228,22229,22230,22231,2223...
Name: 0, dtype: object

Nb moyen d'articles pertinents : 6.53

Top catégories :
category
Famille               272
Logement              238
Argent                141
Justice               121
Etrangers              50
Protection sociale     35
Travail                29
Name: count, dtype: int64


In [13]:
corpus_map = {str(a["id"]): a["article"] for a in corpus}

def parse_ids(raw):
    if isinstance(raw, str):
        return [a.strip() for a in raw.split(",")]
    return [str(a) for a in raw]

def build_qa(questions, split_name):
    examples = []
    for q in questions:
        ids   = parse_ids(q["article_ids"])
        texts = [corpus_map[i] for i in ids if i in corpus_map]
        if texts:
            examples.append({
                "question":    q["question"],
                "answer":      texts[0],
                "article_ids": ids,
                "articles":    texts[:3],
                "category":    q["category"],
            })
    print(f"{split_name} : {len(examples)} paires QA")
    return examples

qa_train = build_qa(q_train, "Train")
qa_test  = build_qa(q_test,  "Test")

print(f"\nExemple :")
print(f"Q : {qa_train[0]['question']}")
print(f"R : {qa_train[0]['answer'][:200]}...")
print(f"IDs: {qa_train[0]['article_ids'][:3]}")

Train : 886 paires QA
Test : 222 paires QA

Exemple :
Q : Je suis travailleur salarié(e). Puis-je refuser de faire des heures supplémentaires ou de travailler de nuit ?
R : Les dispositions du présent titre s'appliquent aux employeurs et aux travailleuses visés à l'article 1er de la loi sur le travail du 16 mars 1971.Elles s'appliquent notamment aux travailleuses visées ...
IDs: ['22225', '22226', '22227']


In [14]:
import json, os

os.makedirs("data/processed", exist_ok=True)

with open("data/processed/qa_train.json", "w", encoding="utf-8") as f:
    json.dump(qa_train, f, ensure_ascii=False, indent=2)

with open("data/processed/qa_test.json", "w", encoding="utf-8") as f:
    json.dump(qa_test, f, ensure_ascii=False, indent=2)

print(f"qa_train.json : {len(qa_train)} exemples")
print(f"qa_test.json  : {len(qa_test)} exemples")

qa_train.json : 886 exemples
qa_test.json  : 222 exemples


In [15]:
from google.colab import userdata
import os

os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")

!git clone https://$GITHUB_TOKEN@github.com/BartoszGOAT/legal-ai.git /content/repo
!cp data/processed/qa_train.json /content/repo/data/processed/
!cp data/processed/qa_test.json  /content/repo/data/processed/

%cd /content/repo
!git config user.email "bartekkonior08@gmail.com"
!git config user.name "BartoszGOAT"
!git add data/processed/
!git commit -m "data: paires QA train/test depuis BSARD"
!git push
%cd /content

Cloning into '/content/repo'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 19 (delta 0), reused 6 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 4.76 KiB | 4.76 MiB/s, done.
/content/repo
[main 432350b] data: paires QA train/test depuis BSARD
 2 files changed, 19147 insertions(+)
 create mode 100644 data/processed/qa_test.json
 create mode 100644 data/processed/qa_train.json
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 1.07 MiB | 2.27 MiB/s, done.
Total 6 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/BartoszGOAT/legal-ai.git
   98f4f7f..432350b  main -> main
/content
